# Amazon Bedrock Converse API

This notebook demonstrates the Amazon Bedrock Converse API, which provides a unified interface for interacting with foundation models. The Converse API simplifies multi-turn conversations and enables advanced features like prompt management and template versioning.

## Learning Objectives

By the end of this notebook, you will understand:
- How to use the Converse API for multi-turn conversations
- System prompt configuration and message management
- Prompt template creation and management with Bedrock Agent
- Variable substitution and template versioning
- Production patterns for conversation-based applications

## Key Concepts

- **Converse API**: Unified interface for foundation model interactions
- **System Prompts**: Instructions that define the assistant's behavior
- **Message History**: Maintaining context across conversation turns
- **Prompt Templates**: Reusable, parameterized prompts for consistent behavior
- **Inference Configuration**: Model parameters that control response generation

## Service Initialization and Configuration

This section establishes connections to AWS services required for conversation management and prompt template operations.

### Key Components:

- **Bedrock Runtime**: Handles model invocation and conversation management
- **Bedrock Agent**: Manages prompt templates, versions, and optimization
- **Nova Micro Model**: AWS foundation model optimized for conversational tasks

### Service Integration Benefits:

- **Unified API**: Single interface for different foundation models
- **Template Management**: Centralized prompt governance and versioning
- **Performance Optimization**: Built-in caching and routing capabilities

In [7]:
import boto3
import json

bedrock = boto3.client(service_name='bedrock-runtime')
bedrock_agent = boto3.client(service_name='bedrock-agent')

MODEL_ID = "amazon.nova-micro-v1:0"

## Multi-Turn Conversation Implementation

This section demonstrates how to build conversational AI applications using the Converse API. The implementation shows proper message management, system prompt configuration, and context preservation across multiple interactions.

### Conversation Architecture:

- **System Prompts**: Define the assistant's role and behavioral constraints
- **Message Array**: Maintains conversation history for context preservation
- **Inference Configuration**: Controls response generation parameters
- **Response Processing**: Extracts and formats model outputs

### Key Implementation Patterns:

- **Stateful Conversations**: Message history accumulates with each turn
- **Role-Based Messaging**: Clear distinction between user and assistant messages
- **Temperature Control**: Balances creativity and consistency in responses

In [10]:
temperature = 0.7

inference_config = {"temperature": temperature}

system_prompts = [{"text": "You are a virtual travel assistant that suggests destinations based on user preferences."
                + "Only return destination names and a brief description."}]

messages = []

message_1 = {
    "role": "user",
    "content": [{"text": "Create a list of 3 travel destinations."}]
}

messages.append(message_1)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

def print_response(response):
    model_response = response.get('output', {}).get('message', {}).get('content', [{}])[0].get('text', '')
    print("✈️ Your suggested travel destinations:")
    print(model_response)

print_response(response)

✈️ Your suggested travel destinations:
1. **Kyoto, Japan**
   - Experience traditional Japanese culture, historic temples, and beautiful gardens.

2. **Cape Town, South Africa**
   - Enjoy stunning beaches, scenic mountains, and vibrant cultural experiences.

3. **Santorini, Greece**
   - Discover breathtaking sunsets, charming white-washed villages, and crystal-clear waters.


## Context Evolution and Conversation Flow

The following cells demonstrate how conversation context evolves as new messages are added to the message history. Each interaction builds upon previous exchanges, showing how the model maintains awareness of earlier requests and constraints.

### Context Management Benefits:

- **Coherent Conversations**: Responses consider full conversation history
- **Constraint Propagation**: Earlier instructions influence subsequent responses
- **Personalization**: Conversations adapt to user preferences over time

### Production Considerations:

- **Token Limits**: Monitor conversation length to avoid context window overflow
- **Memory Management**: Implement conversation summarization for long sessions
- **State Persistence**: Store conversation history for session continuity

In [11]:
message_2 = {
        "role": "user",
        "content": [{"text": "Only suggest travel locations that are no more than one short flight away."}]
}

messages.append(message_2)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

print_response(response)

✈️ Your suggested travel destinations:
1. **Orlando, Florida**  
   Experience world-famous theme parks like Disney World and Universal Studios.

2. **Miami, Florida**  
   Enjoy vibrant nightlife, beautiful beaches, and rich cultural experiences.

3. **San Diego, California**  
   Discover stunning coastlines, famous zoos, and a laid-back lifestyle.


## Geographic Context and Specialization

This interaction demonstrates how the model can adapt its responses based on geographic context while maintaining conversation continuity. The system shows how conversational AI can provide specialized knowledge while respecting earlier constraints.

### Contextual Adaptation Patterns:

- **Geographic Awareness**: Model understands regional context and preferences
- **Cultural Sensitivity**: Responses adapt to local customs and expectations
- **Constraint Inheritance**: Previous limitations continue to influence responses

### Enterprise Applications:

- **Localized Recommendations**: Tailor suggestions to specific regions or markets
- **Compliance Awareness**: Adapt responses to local regulations and requirements
- **Cultural Customization**: Modify communication style for different audiences

In [12]:
message_3 = {
        "role": "user",
        "content": [{"text": "Great places to visit in Brazil's northeast region"}]
}

messages.append(message_3)

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    system=system_prompts,
    inferenceConfig=inference_config
)

print_response(response)

✈️ Your suggested travel destinations:
1. **Salvador, Bahia**  
   A vibrant city known for its colorful colonial architecture, lively street festivals, and stunning beaches.

2. **Recife, Pernambuco**  
   A dynamic coastal city with rich cultural heritage, beautiful beaches, and a lively nightlife.

3. **Fortaleza, Ceará**  
   Famous for its pristine beaches, modern architecture, and a rich blend of Afro-Brazilian culture.


## Prompt Template Creation and Management

This section demonstrates advanced prompt management using Bedrock Agent's template system. Prompt templates enable consistent behavior across applications while supporting variable substitution and version control.

### Template Architecture Components:

- **Template Variants**: Different versions for A/B testing and optimization
- **Input Variables**: Parameterized prompts for dynamic content injection
- **Inference Configuration**: Model-specific parameters embedded in templates
- **System Instructions**: Detailed behavioral guidelines and constraints

### Enterprise Benefits:

- **Governance**: Centralized control over AI behavior and responses
- **Consistency**: Standardized prompts across different applications
- **Optimization**: A/B testing and performance measurement capabilities
- **Compliance**: Audit trails and version control for regulatory requirements

### Template Design Patterns:

The travel agent template demonstrates several important patterns:
- **Structured Evaluation**: Clear criteria for consistent decision-making
- **Risk Assessment**: Built-in safety and compliance checks
- **Variable Substitution**: Dynamic content insertion through placeholders

In [13]:
try:
    response = bedrock_agent.create_prompt(
        name="Travel-Agent-Prompt",
        description="Checks if all trip information has been provided.",
        variants=[
            { 
                "name": "Variant1",
                "modelId": MODEL_ID,
                "templateType": "CHAT",
                "inferenceConfiguration": {
                    "text": {
                        "temperature": 0.4
                    }
                },
                "templateConfiguration": { 
                    "chat": {
                        'system': [ 
                            {
                                "text": """You are a travel agent evaluating trip requests for custom itineraries. 
                                Review the message carefully and answer YES or NO to the following screening questions. 
                                Be strict—if any detail is missing or unclear, answer NO.

                                A) Is the destination clearly stated?
                                B) Are the travel dates within a reasonable range (not last−minute or over a year away)?
                                C) Does the request avoid high−risk or restricted activities (e.g., extreme sports, off−grid travel)?
                                D) Is there any mention of a valid passport or travel documentation?
                                E) Is there enough information to follow up with a proposed itinerary?"""
                            }
                        ],
                        'messages': [{
                            'role': 'user',
                            'content': [ 
                                {
                                    'text': "Trip request: {{event_request}}"
                                }
                            ]
                        }],
                        'inputVariables' : [
                            { 'name' : 'event_request'}
                        ]
                    }
                }
        }]
    )
    print("Created!")
    prompt_arn = response.get("arn")
except bedrock.exceptions.ConflictException as e:
    print("Already exists!")
    response = bedrock.list_prompts()
    prompt = next((prompt for prompt in response['promptSummaries'] if prompt['name'] == "TripBooker_xyz"), None)
    prompt_arn = prompt['arn']

prompt_arn

Created!


'arn:aws:bedrock:us-east-1:206204551974:prompt/FJ6HMMOF87'

## Template Execution and Variable Substitution

This section demonstrates how to execute prompt templates with variable substitution. The pattern shows how templates can be invoked like functions, with dynamic content injected through variables.

### Execution Patterns:

- **Template Invocation**: Using template ARN as model identifier
- **Variable Binding**: Mapping input data to template placeholders
- **Structured Output**: Consistent response format based on template design

### Production Implementation:

- **Template Registry**: Centralized catalog of available templates
- **Version Management**: Deploy and rollback template versions
- **Performance Monitoring**: Track template usage and effectiveness
- **Access Control**: Manage permissions for template creation and execution

In [14]:
response = bedrock.converse(
    modelId=prompt_arn,
    promptVariables={
        'event_request': {
            'text': """
                Hi there! I'm planning a trip to Italy with my partner and would love some help organizing the itinerary. We're hoping to travel between September 10–20 this year, ideally flying into Recife and spending a few days in João Pessoa and Natal as well. We'd love recommendations on tours, cultural sites, and good local restaurants. We're not interested in anything risky like skydiving or hiking remote trails — just want a relaxing and enriching experience. We both have valid passports. Let me know what other details you need!
                """
        }
    },
)
print(response['output']['message']['content'][0]['text'])

A) Is the destination clearly stated?
YES

B) Are the travel dates within a reasonable range (not last−minute or over a year away)?
YES

C) Does the request avoid high−risk or restricted activities (e.g., extreme sports, off−grid travel)?
YES

D) Is there any mention of a valid passport or travel documentation?
YES

E) Is there enough information to follow up with a proposed itinerary?
YES

Answer: YES


## Production Implementation Patterns

When deploying Converse API applications in production environments, several architectural patterns and operational considerations become essential:

### Conversation Management

- **Session Storage**: Persist conversation history in databases or caches
- **Context Compression**: Summarize long conversations to manage token limits
- **Memory Optimization**: Implement sliding window or hierarchical memory
- **State Synchronization**: Handle concurrent access to conversation state

### Template Governance

- **Version Control**: Implement CI/CD pipelines for template deployment
- **A/B Testing**: Compare template variants for optimization
- **Performance Metrics**: Monitor response quality and user satisfaction
- **Rollback Strategies**: Quick recovery from problematic template versions

### Security and Compliance

- **Input Validation**: Sanitize user inputs and template variables
- **Output Filtering**: Implement content moderation and safety checks
- **Audit Logging**: Track all interactions for compliance and debugging
- **Access Control**: Manage permissions for template and model access

### Scalability Considerations

- **Load Balancing**: Distribute requests across multiple regions
- **Caching Strategies**: Cache frequent responses and template results
- **Rate Limiting**: Implement throttling to prevent abuse
- **Cost Optimization**: Monitor token usage and optimize prompt efficiency

## Key Learnings and Best Practices

This notebook demonstrates several critical concepts for building production-ready conversational AI applications:

### Converse API Benefits

1. **Unified Interface**: Single API for multiple foundation models
2. **Conversation Management**: Built-in support for multi-turn interactions
3. **System Integration**: Seamless integration with other AWS services
4. **Performance Optimization**: Automatic routing and caching capabilities

### Template Management Advantages

1. **Consistency**: Standardized behavior across applications
2. **Governance**: Centralized control and version management
3. **Optimization**: A/B testing and performance measurement
4. **Compliance**: Audit trails and regulatory compliance support

### Implementation Patterns

1. **Message Management**: Proper conversation state handling
2. **Context Preservation**: Maintaining coherent multi-turn conversations
3. **Variable Substitution**: Dynamic content injection through templates
4. **Error Handling**: Graceful degradation and recovery strategies

### Next Steps

To extend this implementation:
- Implement conversation persistence and session management
- Add content moderation and safety guardrails
- Create template optimization and A/B testing workflows
- Integrate with enterprise systems and databases
- Implement comprehensive monitoring and analytics